In [3]:
import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm  # For progress bar
from multiprocessing import Pool

import importlib
importlib.reload(utils)


<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [4]:
import pickle

with open('dataset_info_benchmark.pkl', 'rb') as f:
    dataset_info = pickle.load(f)

In [2]:
dataset_info[1]

{'dataset_name': 'benchmark/datasets_90_csv\\1050e533beb01e7b467e438007d7234eed6f60870b36f644c2e7d2bc02f2669f.text.csv',
 'data_profile': 'The key data profile information for the dataset 1050e533beb01e7b467e438007d7234eed6f60870b36f644c2e7d2bc02f2669f.text includes:\n**FIPS**: Data is of type integer. There are 3141 unique values. This column is numeric. Mean: 30411.40727143312, Max: 56045, Min: 1001. Coverage spans from 0 to 55095.0. \n**Year**: Data is of type text. There are 17 unique values. This column is numeric. Mean: 2007.0006930526158, Max: 2015, Min: 1999. \n**State**: Data is of type text. There are 51 unique values. Top 3 frequent values: Texas, Georgia, Virginia. \n**ST**: Data is of type text. There are 51 unique values. Top 3 frequent values: TX, GA, VA. \n**FIPS State**: Data is of type integer. There are 51 unique values. This column is numeric. Mean: 30.307696630265795, Max: 56, Min: 1. Coverage spans from 0 to 55.0. \n**County**: Data is of type text. There are 3141

A questo punto ci serve definire un vector db da riempire con le varie instructions

In [3]:
%pip install -q onnxruntime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import chromadb

In [6]:
try:
    chroma_client = chromadb.PersistentClient()
    chroma_client.get_settings().allow_reset = True
    #chroma_client.reset()
except ValueError as e:
    #print(f"Resetting existing Chroma instance: {e}")
    #chroma_client.reset()
    print("ok")

embedding_function = utils.OpenAIEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="instructions_benchmark",
    embedding_function=embedding_function
)

ok


In [7]:
for dataset in dataset_info:
    instructions = dataset.get("instructions", {}).get("queries", [])
    for i, instruction in enumerate(instructions):
        collection.add(
            documents=[instruction["query"]],
            ids=[f"instruction_{i}_{dataset.get('dataset_name')}"],
            metadatas={
                "dataset": dataset.get("dataset_name"),
                "semantic_profile": dataset.get("semantic_profile"),
                "data_profile": dataset.get("data_profile")
            }
        )

In [8]:
def query_to_vector_db(query_text, n=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n
    )

    print(f"\nQuery: {query}")
    print(f"Top {n} results:")
    for i, doc in enumerate(results['documents'][0]):
        print(f"{i+1}. {doc} (ID: {results['ids'][0][i]}, Distance: {results['distances'][0][i]:.4f})")

In [9]:
query = "I need data which contains information about the nonmarital births in the world"

query_to_vector_db(query, 10)


Query: I need data which contains information about the nonmarital births in the world
Top 10 results:
1. Show me data on the percentage of nonmarital births from 1970 to 2015. (ID: instruction_2_benchmark/datasets_90_csv\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv, Distance: 0.5470)
2. Find data on nonmarital births with percentage values ranging from 0% to 50%. (ID: instruction_16_benchmark/datasets_90_csv\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv, Distance: 0.5540)
3. Show me data that analyzes the trend of nonmarital births over the years. (ID: instruction_11_benchmark/datasets_90_csv\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv, Distance: 0.5608)
4. Find a dataset containing statistics on nonmarital births over multiple years. (ID: instruction_0_benchmark/datasets_90_csv\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv, Distance: 0.5650)
5. Search for datasets offer